# Minimal Model Comparison: Quick Execution Verification

**Purpose:** Verify that LogisticGLM and GRU models execute correctly with the utility function.

**Runtime:** ~2-3 minutes (vs 10-15 minutes for full test)

**Configuration:**
- Models: LogisticGLM, GRU (skipping XGBoost to reduce dependencies)
- Training sizes: [50, 100]
- Bootstrap sizes: [25]
- Iterations: 3 per config
- **Total: 2 × 2 × 1 × 3 = 12 evaluations**

After verifying execution, use `test_utility_at_scales.ipynb` for full evaluation.

In [ ]:
import sys, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Add rebuild directory to imports
sys.path.insert(0, '.')
logging.basicConfig(level=logging.WARNING)

from data_loader import (
    load_physionet_files,
    add_hours_until_sepsis,
    split_patients_by_status,
    get_rows_for_patients,
)
from bootstrap import BootstrapResampler
from models import LogisticGLM, GRUModel
from training import BootstrapEvaluator, extract_Xy

print("✓ All imports successful")

## 1. Configuration

Minimal experimental grid for quick verification:

In [ ]:
# MINIMAL configuration for quick execution
TRAIN_SIZES = [50, 100]        # 2 training sizes (skip 200, 300)
BOOTSTRAP_SIZES = [25]         # 1 bootstrap size (skip 50, 100)
N_ITER = 3                     # 3 iterations per config (vs 10 in full)
RANDOM_STATE = 42

print(f"Training sizes: {TRAIN_SIZES}")
print(f"Bootstrap sizes: {BOOTSTRAP_SIZES}")
print(f"Iterations per config: {N_ITER}")
print(f"Total evaluations: {len(TRAIN_SIZES)} × {len(BOOTSTRAP_SIZES)} × {N_ITER} = {len(TRAIN_SIZES) * len(BOOTSTRAP_SIZES) * N_ITER}")

## 2. Load Data

In [ ]:
# Load PhysioNet PSV files
DATA_DIR = Path('../data/physionet_sepsis')

print('Loading PSV files …')
raw_df = load_physionet_files(DATA_DIR)
print(f'  {raw_df["patient_id"].nunique():,} patients · {len(raw_df):,} rows')

print('Computing hours_until_sepsis …')
df = add_hours_until_sepsis(raw_df, keep_post_onset=True)

# Sanity check
n_septic = df['hours_until_sepsis'].notna().sum()
print(f'  {n_septic:,} septic rows across {df["patient_id"].nunique():,} patients')
print(f'\n✓ Data loaded successfully')

## 3. Run Minimal Evaluation Grid

Quick iteration through 2 models × 2 train sizes × 1 boot size × 3 iterations = 12 total runs

In [ ]:
results = []
config_num = 0
total_configs = len(TRAIN_SIZES) * len(BOOTSTRAP_SIZES) * N_ITER

model_configs = [
    ('LogisticGLM', LogisticGLM(C=0.01)),
    ('GRU', GRUModel(hidden_size=64, num_layers=1, dropout=0.2, epochs=5))  # Reduced for speed
]

for model_name, model_template in model_configs:
    print(f"\n{'='*70}")
    print(f"MODEL: {model_name}")
    print(f"{'='*70}")
    
    for train_size in TRAIN_SIZES:
        print(f"\n  Train size: {train_size}")
        
        # Split patients
        train_pids, boot_pids = split_patients_by_status(
            df, n_train_patients=train_size,
            random_state=RANDOM_STATE, stratify_by_sepsis=True
        )
        train_df = get_rows_for_patients(df, train_pids)
        
        for boot_size in BOOTSTRAP_SIZES:
            print(f"    Boot size: {boot_size}")
            
            # Create fresh model instance for this config
            if model_name == 'LogisticGLM':
                model = LogisticGLM(C=0.01)
            else:
                model = GRUModel(hidden_size=64, num_layers=1, dropout=0.2, epochs=5)
            
            # Create evaluator (fits model on training set)
            evaluator = BootstrapEvaluator(
                model=model,
                train_df=train_df,
                label_column='SepsisLabel',
                patient_id_column='patient_id'
            )
            
            # Bootstrap resampler
            resampler = BootstrapResampler(
                bootstrap_pool_patient_ids=boot_pids,
                full_df=df,
                n_iterations=N_ITER,
                bootstrap_sample_size=boot_size,
                random_state=RANDOM_STATE
            )
            
            # Evaluate on each bootstrap sample
            for iter_idx in range(N_ITER):
                config_num += 1
                print(f"      [{config_num:2d}/{total_configs}] Iteration {iter_idx+1}... ", end='', flush=True)
                
                try:
                    _, boot_df = resampler.generate_iteration(iter_idx)
                    
                    # Evaluate on bootstrap sample
                    m = evaluator.evaluate_iteration(
                        boot_df, iter_idx,
                        compute_per_group=True,
                        group_column='Gender'
                    )
                    
                    # Extract utility by group
                    utility_female = np.nan
                    utility_male = np.nan
                    if 'per_group' in m:
                        if 0 in m['per_group']:
                            utility_female = m['per_group'][0].get('utility', np.nan)
                        if 1 in m['per_group']:
                            utility_male = m['per_group'][1].get('utility', np.nan)
                    
                    results.append({
                        'model': model_name,
                        'train_size': train_size,
                        'boot_size': boot_size,
                        'iteration': iter_idx,
                        'utility': m.get('utility', np.nan),
                        'auroc': m.get('auroc', np.nan),
                        'recall': m.get('recall', np.nan),
                        'f1': m.get('f1', np.nan),
                        'accuracy': m.get('accuracy', np.nan),
                        'utility_female': utility_female,
                        'utility_male': utility_male,
                    })
                    
                    print(f"✓ Utility: {m.get('utility', np.nan):.4f}")
                    
                except Exception as e:
                    print(f"✗ ERROR: {str(e)[:50]}")
                    results.append({
                        'model': model_name,
                        'train_size': train_size,
                        'boot_size': boot_size,
                        'iteration': iter_idx,
                        'utility': np.nan,
                        'auroc': np.nan,
                        'recall': np.nan,
                        'f1': np.nan,
                        'accuracy': np.nan,
                        'utility_female': np.nan,
                        'utility_male': np.nan,
                    })

print(f"\n\n{'='*70}")
print(f"EVALUATION COMPLETE")
print(f"{'='*70}")

## 4. Results Summary

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(results)

print(f"\nTotal results collected: {len(results_df)}")
print(f"\nResults dataframe shape: {results_df.shape}")
print(f"\nFirst 10 rows:\n")
print(results_df.head(10).to_string())

## 5. Best Configurations by Model

In [ ]:
print("\nBEST CONFIGURATIONS BY MODEL")
print("="*80)

for model_name in sorted(results_df['model'].unique()):
    model_results = results_df[results_df['model'] == model_name]
    
    if model_results['utility'].isna().all():
        print(f"\n{model_name}: No valid results")
        continue
    
    best_idx = model_results['utility'].idxmax()
    best = model_results.loc[best_idx]
    
    print(f"\n{model_name}:")
    print(f"  Train size: {best['train_size']:.0f} patients")
    print(f"  Boot size: {best['boot_size']:.0f} patients/sample")
    print(f"  Utility: {best['utility']:.4f}")
    print(f"  AUROC: {best['auroc']:.4f}")
    print(f"  Recall: {best['recall']:.4f}")
    print(f"  F1: {best['f1']:.4f}")
    
    if not np.isnan(best['utility_female']) and not np.isnan(best['utility_male']):
        gap = abs(best['utility_female'] - best['utility_male'])
        print(f"  Gender utility gap: {gap:.4f}")

## 6. Aggregated Summary

In [ ]:
# Summary by model and train size
summary_data = []

for model in sorted(results_df['model'].unique()):
    for train_size in sorted(results_df['train_size'].unique()):
        subset = results_df[(results_df['model'] == model) & (results_df['train_size'] == train_size)]
        
        summary_data.append({
            'model': model,
            'train_size': train_size,
            'utility_mean': subset['utility'].mean(),
            'utility_std': subset['utility'].std(),
            'auroc_mean': subset['auroc'].mean(),
            'auroc_std': subset['auroc'].std(),
            'recall_mean': subset['recall'].mean(),
            'f1_mean': subset['f1'].mean(),
        })

summary_df = pd.DataFrame(summary_data)

print("\nSUMMARY BY MODEL AND TRAINING SIZE")
print("="*80)
print(summary_df.round(4).to_string(index=False))

# Save summary
summary_df.to_csv('utility_model_comparison_summary_minimal.csv', index=False)
print("\n✓ Summary saved to utility_model_comparison_summary_minimal.csv")

## 7. Visualizations

In [ ]:
# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# 1. Utility by Model and Training Size
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Utility
for model in sorted(results_df['model'].unique()):
    model_data = results_df[results_df['model'] == model]
    train_sizes = sorted(model_data['train_size'].unique())
    utilities = [model_data[model_data['train_size'] == ts]['utility'].mean() for ts in train_sizes]
    axes[0].plot(train_sizes, utilities, marker='o', label=model, linewidth=2, markersize=8)

axes[0].set_xlabel('Training Size (patients)', fontsize=12)
axes[0].set_ylabel('Mean Utility', fontsize=12)
axes[0].set_title('Utility Score vs Training Size', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# AUROC
for model in sorted(results_df['model'].unique()):
    model_data = results_df[results_df['model'] == model]
    train_sizes = sorted(model_data['train_size'].unique())
    aurocs = [model_data[model_data['train_size'] == ts]['auroc'].mean() for ts in train_sizes]
    axes[1].plot(train_sizes, aurocs, marker='s', label=model, linewidth=2, markersize=8)

axes[1].set_xlabel('Training Size (patients)', fontsize=12)
axes[1].set_ylabel('Mean AUROC', fontsize=12)
axes[1].set_title('AUROC vs Training Size', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison_metrics_minimal.png', dpi=150, bbox_inches='tight')
print("✓ Saved: model_comparison_metrics_minimal.png")
plt.show()

In [ ]:
# 2. Distribution Box Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Utility distributions
sns.boxplot(data=results_df, x='model', y='utility', ax=axes[0])
axes[0].set_title('Utility Score Distributions', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Utility', fontsize=12)
axes[0].set_xlabel('Model', fontsize=12)

# AUROC distributions
sns.boxplot(data=results_df, x='model', y='auroc', ax=axes[1])
axes[1].set_title('AUROC Distributions', fontsize=13, fontweight='bold')
axes[1].set_ylabel('AUROC', fontsize=12)
axes[1].set_xlabel('Model', fontsize=12)

plt.tight_layout()
plt.savefig('model_distributions_minimal.png', dpi=150, bbox_inches='tight')
print("✓ Saved: model_distributions_minimal.png")
plt.show()

In [ ]:
# 3. Gender Fairness Gap
fig, ax = plt.subplots(figsize=(10, 5))

results_df['gender_gap'] = (results_df['utility_female'] - results_df['utility_male']).abs()

fairness_data = results_df.groupby(['model', 'train_size'])['gender_gap'].mean().reset_index()

# Pivot and plot
if len(fairness_data) > 0:
    pivot = fairness_data.pivot(index='model', columns='train_size', values='gender_gap')
    pivot.plot(kind='bar', ax=ax, width=0.7)
    
    ax.set_title('Gender Utility Gap by Model and Training Size', fontsize=13, fontweight='bold')
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel('|Utility (Female) - Utility (Male)|', fontsize=12)
    ax.legend(title='Train Size', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    plt.xticks(rotation=0)
    
    plt.tight_layout()
    plt.savefig('fairness_gender_gap_minimal.png', dpi=150, bbox_inches='tight')
    print("✓ Saved: fairness_gender_gap_minimal.png")
    plt.show()
else:
    print("No fairness data available")

## 8. Save Results CSV

In [ ]:
# Save full results
results_df.to_csv('utility_model_comparison_results_minimal.csv', index=False)

print("\n" + "="*70)
print("EXECUTION VERIFICATION COMPLETE")
print("="*70)
print(f"\n✓ Total evaluations completed: {len(results_df)}")
print(f"✓ Models tested: {', '.join(sorted(results_df['model'].unique()))}")
print(f"✓ CSV results saved: utility_model_comparison_results_minimal.csv")
print(f"✓ Summary saved: utility_model_comparison_summary_minimal.csv")
print(f"\nFigures generated:")
print("  - model_comparison_metrics_minimal.png")
print("  - model_distributions_minimal.png")
print("  - fairness_gender_gap_minimal.png")
print(f"\nNext: Run test_utility_at_scales.ipynb with full config for production results.")
print("="*70)